<a href="https://colab.research.google.com/github/dianamtzv2319-dev/Desarrollo-IA/blob/main/segundo_avance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Descargar recursos necesarios de NLTK
nltk.download('punkt')
nltk.download('stopwords')
# Si marca error al descargar en Colab, a veces es necesario agregar nltk.download('punkt_tab')
nltk.download('punkt_tab', quiet=True)

# 2. CONJUNTO DE DATOS TEXTUAL (Ejemplos de entrenamiento)
datos = [
    ("hola buenas tardes", "saludo"), ("qué tal, buenos días", "saludo"),
    ("hola, disculpe", "saludo"), ("hola, una duda", "saludo"),
    ("cuáles son los requisitos para adoptar", "requisitos"),
    ("qué necesito para llevarme un perrito", "requisitos"),
    ("cómo es el proceso de adopción", "requisitos"),
    ("quiero adoptar qué hago", "requisitos"),
    ("ya los entregan operados", "esterilizacion"),
    ("están esterilizados los cachorros", "esterilizacion"),
    ("qué onda con la esterilización", "esterilizacion"),
    ("los perros están castrados", "esterilizacion"),
    ("me pasas el número de teléfono", "contacto"),
    ("dónde me comunico para informes", "contacto"),
    ("quiero hablar con alguien pasame el contacto", "contacto"),
    ("a qué número llamo", "contacto"),
    ("qué tamaño tiene este perro en la foto", "clasificar_imagen"),
    ("analiza esta imagen por favor", "clasificar_imagen"),
    ("te mando una foto del perrito", "clasificar_imagen"),
    ("de qué tamaño es", "clasificar_imagen")
]

# Crear un DataFrame (Tabla virtual)
df = pd.DataFrame(datos, columns=['texto', 'intencion'])

# 3. PREPROCESAMIENTO DE TEXTO (NLP)
stemmer = SnowballStemmer('spanish')
stop_words = set(stopwords.words('spanish'))

def preprocesar_texto(texto):
    # A) Normalización (minúsculas)
    texto = texto.lower()
    # B) Tokenización (separar palabras)
    tokens = word_tokenize(texto)
    # Conservar solo palabras (eliminar signos de puntuación)
    tokens = [word for word in tokens if word.isalpha()]
    # C) Eliminación de Stopwords (palabras vacías)
    tokens = [word for word in tokens if word not in stop_words]
    # D) Stemming (reducir palabras a su raíz)
    tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

# Aplicar el preprocesamiento a nuestra columna de texto
df['texto_procesado'] = df['texto'].apply(preprocesar_texto)
print("Ejemplo de texto preprocesado:")
print(df[['texto', 'texto_procesado']].head())
print("-" * 50)

# 4. REPRESENTACIÓN DEL TEXTO (TF-IDF)
vectorizador = TfidfVectorizer()
X = vectorizador.fit_transform(df['texto_procesado'])
y = df['intencion']

# 5. ENTRENAMIENTO DEL CLASIFICADOR (SVM - Máquinas de Vectores de Soporte)
# Dividimos los datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clasificador = SVC(kernel='linear')
clasificador.fit(X_train, y_train)

# 6. EVALUACIÓN DEL DESEMPEÑO
y_pred = clasificador.predict(X_test)

print("\n--- RESULTADOS DEL CLASIFICADOR DE TEXTO ---")
print(f"Accuracy (Exactitud): {accuracy_score(y_test, y_pred)}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Ejemplo de texto preprocesado:
                                    texto      texto_procesado
0                      hola buenas tardes        hol buen tard
1                    qué tal, buenos días        tal buen dias
2                          hola, disculpe          hol disculp
3                          hola, una duda              hol dud
4  cuáles son los requisitos para adoptar  cual requisit adopt
--------------------------------------------------

--- RESULTADOS DEL CLASIFICADOR DE TEXTO ---
Accuracy (Exactitud): 0.5

Reporte de Clasificación:
                   precision    recall  f1-score   support

clasificar_imagen       0.00      0.00      0.00         1
         contacto       1.00      1.00      1.00         1
       requisitos       0.00      0.00      0.00         0
           saludo       1.00      0.50      0.67         2

         accuracy                           0.50         4
        macro avg       0.50      0.38      0.42         4
     weighted avg       0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ==========================================
# 1. PREPARACIÓN DE LOS DATOS REALES
# ==========================================
print("Preparando el conjunto de datos para la Red Neuronal...")
base_dir = 'dataset_perros'
categorias = ['pequenos', 'grandes']

# Normalizamos las imágenes y separamos el 20% para pruebas
datagen = ImageDataGenerator(validation_split=0.2, rescale=1./255)

# Datos de entrenamiento (80%)
train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=8,
    class_mode='binary',
    subset='training',
    shuffle=False
)

# Datos de validación/prueba (20%)
val_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=8,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

# ==========================================
# 2. ARQUITECTURA DEL MODELO (RESNET50 - TRANSFER LEARNING)
# ==========================================
print("\nCargando la arquitectura ResNet50...")
# Cargamos el modelo base ResNet50 pre-entrenado
modelo_base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Congelamos las capas base para no perder el aprendizaje previo
modelo_base.trainable = False

# Añadimos nuestras propias capas finales para clasificar (Pequeño vs Grande)
modelo_visual = Sequential([
    modelo_base,
    GlobalAveragePooling2D(),
    Dense(1, activation='sigmoid') # Sigmoid porque son solo 2 clases
])

modelo_visual.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# ==========================================
# 3. ENTRENAMIENTO DEL MODELO
# ==========================================
print("\nIniciando entrenamiento del modelo visual...")
historia = modelo_visual.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,  # Subimos a 5 épocas ya que son fotos reales para que aprenda mejor
    verbose=1
)

# ==========================================
# 4. EVALUACIÓN Y MÉTRICAS (LO QUE PIDE LA RÚBRICA)
# ==========================================
print("\n--- CALCULANDO MÉTRICAS DE DESEMPEÑO ---")
# Hacemos que el modelo clasifique las imágenes de prueba
y_pred_probs = modelo_visual.predict(val_generator)
y_pred = (y_pred_probs > 0.5).astype(int).reshape(-1) # Convertir probabilidad a 0 o 1
y_real = val_generator.classes

# Métricas exactas requeridas por la rúbrica
exactitud = accuracy_score(y_real, y_pred)
precision = precision_score(y_real, y_pred, zero_division=0)
sensibilidad = recall_score(y_real, y_pred, zero_division=0)
f1 = f1_score(y_real, y_pred, zero_division=0)

print(f"Accuracy (Exactitud): {exactitud:.2f}")
print(f"Precision:            {precision:.2f}")
print(f"Recall (Sensibilidad):{sensibilidad:.2f}")
print(f"F1-score:             {f1:.2f}")

# Matriz de Confusión
print("\nGenerando Matriz de Confusión...")
matriz = confusion_matrix(y_real, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=categorias, yticklabels=categorias)
plt.title('Matriz de Confusión - Clasificador de Perros')
plt.ylabel('Valor Real')
plt.xlabel('Predicción del Modelo')
plt.show()

Preparando el conjunto de datos para la Red Neuronal...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset_perros'

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

# ==========================================
# CHATBOT INTERACTIVO (INTEGRACIÓN MULTIMODAL)
# ==========================================
print("=========================================================")
print(" INICIANDO CHATBOT DE ADOPCIONES V2 (CON MACHINE LEARNING)")
print(" Escribe 'salir' para terminar la conversación.")
print("=========================================================")

while True:
    try:
        entrada = input("Tú: ")
        if entrada.lower() in ['salir', 'adios', 'bye']:
            print("Chatbot: ¡Hasta luego! Gracias por apoyar el rescate animal.")
            break

        # 1. El modelo limpia el texto y predice la intención
        texto_limpio = preprocesar_texto(entrada)
        vector_entrada = vectorizador.transform([texto_limpio])
        intencion = clasificador.predict(vector_entrada)[0]

        # 2. Respuestas según la intención detectada por la IA
        if intencion == 'saludo':
            print("Chatbot: ¡Hola! Bienvenido al asistente virtual inteligente. ¿En qué te puedo ayudar?")

        elif intencion == 'requisitos':
            print("Chatbot: Para adoptar necesitamos que seas mayor de edad. Los requisitos son: copia de INE y comprobante de domicilio.")

        elif intencion == 'esterilizacion':
            print("Chatbot: Cuidamos mucho la salud. Los perros se entregan esterilizados o con promesa de esterilización.")

        elif intencion == 'contacto':
            print("Chatbot: ¡Claro! Para atención directa, comunícate al 729 8799346.")

        elif intencion == 'clasificar_imagen':
            # 3. Integración Visual: Si el usuario pregunta por una foto, activamos la Red Neuronal
            print("Chatbot: ¡Claro! Por favor, escribe la ruta exacta de la foto que quieres que analice (ejemplo: dataset_perros/pequenos/foto.jpg)")
            ruta_img = input("Ruta de la imagen: ")

            try:
                # Cargamos la imagen y la pasamos por ResNet50
                img = image.load_img(ruta_img, target_size=(224, 224))
                img_array = image.img_to_array(img)
                img_array = np.expand_dims(img_array, axis=0) / 255.0

                # Predicción visual
                prediccion = modelo_visual.predict(img_array, verbose=0)

                if prediccion[0][0] > 0.5:
                    print("Chatbot: Viendo la foto, parece un perro de talla GRANDE. ¡Recuerda que necesitará un espacio amplio para jugar!")
                else:
                    print("Chatbot: Viendo la foto, parece un perro de talla PEQUEÑA. ¡Ideal para espacios más reducidos o interiores!")
            except:
                print("Chatbot: Lo siento, no pude encontrar o procesar la imagen en esa ruta. Verifica que el nombre esté bien escrito.")

    except EOFError:
        break